# Phase 1: back half of the registration model with oracle and perturbed seeds

## Description

<!-- Written before the first run (2026-10-23, 11:50). Spec: docs/spec_phase1_experiments.md (user). Criteria drafted by Claude; the spec's [decide] items use its proposed defaults, listed under Limits. -->

**Question.** The registration design (docs/design_2026-10-23_gw-landmarks-displacement-field.md) has a front half that finds candidate correspondences and a back half (steps 3–5) that refines a correspondence field and reads it out. This experiment tests the back half on its own, given oracle or approximate-oracle correspondences, under controlled input (T) and output (h) distortions. Four questions:
1. **What is B worth when transfer is perfect?** The effective-data curve (E2).
2. **Can the read-out turn the true correspondence into that value?** (E3)
3. **How wrong may a seed be before refinement can no longer recover it?** (E4)
4. **How do T and h differ?**

**Setup.**
- **Prior** (`experimental/regpfn/prior.py`). f_A is a BNN draw (pfn-repro), evaluated raw so it extends beyond [0, 1]^d. B's function is f_B(u) = h(f_A(T(u))), and both tasks sample x ∈ [0, 1]^d (N_B = 256 uniform; A has n_A ∈ [0, 20] context points and 128 queries).
  - **T** = Kumaraswamy warp ∘ per-dim scale (a ∈ logU[0.7, 1.4]) ∘ shift (s ∈ U[−0.3, 0.3]).
  - **h** = affine ∘ sinh-arcsinh warp (δ ∈ logU[0.5, 2], ε ∈ U[−1, 1]).
  - **Normalization and noise.** Each task is normalized by min/max of its noise-free function on a dense grid of its own window, onto [0.1, 0.9]. Noise σ_ε ∈ logU[1e-3, 3e-2] is added afterwards and y is clipped to [0, 1].
  - **Cells:** C00 (neither), C10 (T), C01 (h), C11 (both).
  - **Oracle:** B_inA = (T(u), norm_A(h⁻¹(norm_B⁻¹(y_B)))), masked where T(u) ∉ [0, 1]^d. E0 checks: exact to < 5e-7 without noise in all cells, d = 1, 2 (tests/unit/test_regpfn.py; plot grid in the log).
- **E1/E2: A-only PFN.** pfn-repro's PFN (6 layers, d_model 256, 64 uniform bins on [0, 1]), trained here on A's data from this prior, with context lengths up to 20 + 256. E2: NLL on A's queries with A's n_A points plus m random inside-window B_inA points, m ∈ {0, 1, 2, 4, …, 256}, 5 random subsets. Then m* = the m at which this curve reaches a model's NLL (monotone interpolation, same cell and n_A).
- **Back half** (`RegBack`, `experimental/regpfn/model.py`). One seeded slot + the null slot.
  - **Refinement:** a weight-tied GRU. Each step does a distance-biased lookup of A's points at x̂ and gates the update by evidence (no A point near ⇒ no move). Corrections are smoothed over B's own coordinates.
  - **Read-out:** distance-biased attention in A's frame. Mixture with the null slot through a learned weight head.
  - **Loss:** mixture NLL + LUPI field loss Σ_t 0.8^{T−t} |(x̂_t, ŷ_t) − B_inA|² on inside points.
- **Runs** (d = 1, one each, 25 min of training on ulysses' RTX 3090):
  - `aonly` (E1 + E2);
  - `back true, n_iter = 0` (E3);
  - `back perturb, n_iter = 8` and `n_iter = 0` (E4). Training σ ∈ U[0, 0.25] for x and y, each on with probability ½; 30% of x-perturbations are global shifts.
- **Evaluation.** 128 fixed pairs per cell (the same pairs in every run), n_A ∈ {0, 2, 5, 10, 20}. Perturbation ladder σ ∈ {0.02, 0.05, 0.1, 0.2} for x (smooth field), shift, and y.

**What to look for.**
- E1: NLL vs n_A.
- E2: NLL(m) per cell and n_A. Curves should differ across cells only through coverage (inside fraction) and noise distortion; a large difference means a bug.
- E3a: in C00 with the true seed, does the model reach the m ≈ N_B end of the curve, and never fall below A alone?
- E3b: m* per cell with the true seed and no refinement.
- E4, per cell and kind: m* vs σ; field error before and after refinement vs σ; null weight vs σ; n_iter = 8 vs 0.

**Decision rule** (drafted by Claude; d = 1, n_A ∈ {2, 5, 10, 20}):
- **E3a passes** if in C00 the true-seed model's NLL ≤ the A-only NLL (m = 0) + 0.02 at every n_A, and its m* ≥ 128.
- **E3b: the read-out is OK** if m* ≥ 128 (half of N_B) in all four cells; **WEAK** if 16 ≤ m* < 128 in some cell; **FAIL** if m* < 16 in some cell.
- **E4: refinement helps** in a (cell, kind) if NLL(n_iter = 8) < NLL(n_iter = 0) on perturbed seeds for σ ≥ 0.05 at n_A ≥ 5. The **convergence radius** is the largest σ on the ladder at which refinement cuts the field error by ≥ 50% at n_A = 10. The null weight should rise with σ.
- **Stop rule** (spec §9): if E3a fails, E4 is not interpreted.

**Limits.**
- **[decide] defaults, flagged:**
  - BNN: pfn-repro's own ranges, evaluated raw outside [0, 1]^d.
  - T: s_max 0.3, a ∈ [0.7, 1.4], Kumaraswamy log-sd 0.35.
  - h: sinh-arcsinh plus affine. The affine part is invisible after B's min/max normalization.
  - Normalization: per-task min/max onto [0.1, 0.9], noise added after it, clipped to [0, 1].
  - Bar distribution: uniform bins rather than prior quantiles.
  - Model sizes: back half d_model 128, 3 read-out layers; A-only PFN 6 × 256. T_iter = 8, Fourier features (8 octaves).
- **A_inB** is not produced (not needed for the back half).
- **Training budget:** 25 min per model is short, and m* is read off our own A-only PFN, so the ceiling is only as good as that model.
- **Scope:** d = 1 only in this round. 100% related pairs: no unrelated pairs, so the null slot is never right; its weight only measures distrust of a bad seed.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meta_pfn.analysis import load_snapshot, show_header
from meta_pfn.analysis.regpfn import curve, e3_checks, e4_tables, with_m_star

# one run each (ulysses, commit c1f9f3d)
RUNS = {
    "aonly": "results/regpfn/pilot_20261023-115248",       # E1 + E2
    "true0": "results/regpfn/pilot_20261023-115419",       # E3: true seed, n_iter = 0
    "perturb8": "results/regpfn/pilot_20261023-115249",    # E4: perturbed seeds, n_iter = 8
    "perturb0": "results/regpfn/pilot_20261023-115307",    # E4: perturbed seeds, n_iter = 0
}
snaps = {k: load_snapshot(p) for k, p in RUNS.items()}
for k, s in snaps.items():
    print(k, s.runs.status.value_counts().to_dict(), list(s.runs.git_commit.str[:7].unique()))
show_header(snaps["perturb8"])

## E1 / E2: the A-only PFN and the effective-data curves

NLL on A's 128 queries (nats; y on [0, 1]); m = number of B_inA points added to A's n_A points (mean over 5 random subsets; capped by the inside-window count).

In [ ]:
ev_a = snaps["aonly"].table("aonly_eval")
cv = curve(ev_a)
display(cv.loc[cv.m == 0].pivot_table(index="cell", columns="n_a", values="nll").round(3))   # E1: NLL vs n_A
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), sharey=True)
for ax, (cell, h) in zip(axes, cv.groupby("cell")):
    for n, hh in h.groupby("n_a"):
        ax.plot(hh.m + 1, hh.nll, "-o", ms=3, label=f"n_A={n}")
    ax.set_xscale("log", base=2); ax.set_title(cell); ax.set_xlabel("1 + m"); ax.grid(alpha=0.3)
axes[0].set_ylabel("NLL on A queries"); axes[0].legend(fontsize=8); fig.tight_layout()
display(cv.loc[cv.m == 256].pivot_table(index="cell", columns="n_a", values="m_inside").round(1))  # coverage

## E3: the read-out with the true field

In [ ]:
ev_t = with_m_star(snaps["true0"].table("back_eval"), cv)
display(ev_t.pivot_table(index="cell", columns="n_a", values=["nll", "nll_lower", "m_star", "w_null"]).round(3))
e3 = e3_checks(ev_t)
e3

## E4: perturbed seeds, refinement (n_iter = 8) vs read-out only (n_iter = 0)

In [ ]:
ev8 = with_m_star(snaps["perturb8"].table("back_eval"), cv)
ev0 = with_m_star(snaps["perturb0"].table("back_eval"), cv)
helps, radius = e4_tables(ev8, ev0)
display(helps.loc[helps.n_a == 10].pivot_table(index=["cell", "kind"], columns="sigma",
        values=["nll_refined", "nll_readout_only_model", "err_seed", "err_final", "w_null_refined"]).round(3))
display(radius.pivot_table(index="cell", columns="kind", values="radius"))
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, kind in zip(axes, ["x", "shift", "y"]):
    for cell, h in helps.loc[(helps.kind == kind) & (helps.n_a == 10)].groupby("cell"):
        ax.plot(h.sigma, h.m_star_refined.clip(upper=300), "-o", ms=3, label=f"{cell} refined")
        ax.plot(h.sigma, h.m_star_readout_only.clip(upper=300), "--", label=f"{cell} read-out only")
    ax.set_title(f"m* vs σ ({kind}), n_A = 10"); ax.set_xlabel("σ"); ax.grid(alpha=0.3)
axes[0].set_ylabel("m* (effective B points)"); axes[0].legend(fontsize=6); fig.tight_layout()

### Training curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for k in ("true0", "perturb8", "perturb0"):
    t = snaps[k].table("train"); ax.plot(t.seconds / 60, t.nll, label=k)
t = snaps["aonly"].table("train"); ax.plot(t.seconds / 60, t.loss, label="aonly")
ax.set_xlabel("minutes"); ax.set_ylabel("training NLL"); ax.legend(); ax.grid(alpha=0.3)

## Observations

One run each, d = 1, all DONE, commit c1f9f3d (ulysses). Snapshots: `results/regpfn/pilot_20261023-115248` (aonly), `…-115419` (true seed, n_iter = 0), `…-115249` (perturbed, n_iter = 8), `…-115307` (perturbed, n_iter = 0). The evaluation uses the same 128 pairs per cell in every run; NLL on A's 128 queries.

- **E0** (before training; tests + plot grid): B_inA is exact to < 5e-7 without noise in all cells at d = 1, 2. Inside-window fraction: 1.00 (C00, C01), 0.85 (C10, C11; 217 of 256 B points).
- **E1** (A-only PFN, 9400 steps): NLL −0.19 / −0.68 / −1.20 / −1.79 / −2.28 at n_A = 0 / 2 / 5 / 10 / 20 (C00). The other cells agree within 0.13, as expected: A's marginal doesn't depend on the cell.
- **E2** (effective-data curves): smooth and monotone. Adding all of B_inA reaches −2.98 (C00) and −3.03 (C01), whatever n_A is. With T on it reaches −2.64 to −2.94, lower because only 85% of B falls in A's window. So the curves differ across cells only through coverage, the sanity expectation of the spec.
- **E3: the read-out turns the true field into a large gain, but captures only a fraction of B's value.** With the true seed and no refinement, the model reaches NLL −2.63 (C00) at n_A = 0, a gain of 2.4 nats over A alone. Its m* is only 37 (C00), 35 (C01) and 19 (C10, C11), out of 256 points that would bring the A-only PFN to −2.98.
  - m* *falls* with n_A (C00: 38 at n_A = 5, 14 at n_A = 20). The model's NLL barely improves with more A points (−2.69 → −2.59), while the reference does.
  - The mixture weight on the null slot rises to 0.33 at n_A = 20, although every pair is related. log(0.67) alone costs up to 0.4 nats there.
  - The model trained with perturbed seeds (n_iter = 0) is identical on true seeds (m* 37 vs 37): perturbation training didn't cost read-out quality.
- **E4 (descriptive only, see Verdict):**
  - **y-perturbations:** refinement recovers them. The field error falls from 0.20 to 0.065 (C00), 0.09 (C01) and 0.10 (C10) at σ = 0.2, n_A = 10, and NLL improves over the read-out-only model for σ ≥ 0.05 (e.g. C00, σ = 0.2: −2.05 vs −1.76). The convergence radius is ≥ 0.2 for y in C00, C01, C10 (0 in C11).
  - **x- and shift-perturbations:** refinement does *not* recover them (error 0.20 → 0.19; radius 0 in every cell), and the null weight rises with σ (C00 shift: 0.03 → 0.23).
  - **Perfect seeds get worse:** refinement moves correct seeds away (field error 0 → 0.05–0.07 when T is on), so with a perfect seed the refining model is worse than read-out only (m* 23 vs 37 in C00, 13 vs 19 in C10).
  - It trained 35% fewer steps in the same 25 minutes (5500 vs 8500).
  - The no-evidence check holds in the trained model: field change at n_A = 0 is exactly 0.

**Diagnosis.**
1. The learned read-out (d_model 128, 3 layers, 25 min) is a much weaker reader of 256 rendered points than the A-only PFN (6 × 256, trained on contexts up to 276). The spec's E3 premise is that the read-out *is* that reader. The direct fix: use the trained A-only PFN as the seed slot's read-out (frozen, or as the initialization) on A ∪ rendered B. E3 then reaches the curve by construction, and the remaining question is the field.
2. The weight head over-weights the null slot as n_A grows. With 100% related pairs it should learn w_null → 0; it needs a better input than pooled hidden states, e.g. the seeded slot's own predictive score on A's points (the evidence weights of brainstorm §5.3).
3. Refinement fixes values (y) but not positions (x). A position error shows up only as a y-mismatch at A's few points, and the lookup sees no explicit residual. A variant with explicit local residuals and bounded steps (`resid`, `max_step`) ran next (§ Update).

## Verdict

By the pre-registered rules (drafted by Claude):
- **E3a: FAIL.** In C00 with the true seed, the NLL is below A alone at every n_A, but m* ≥ 128 fails (m* = 14–41).
- **E3b: FAIL.** The minimum m* per cell is 13.7 / 12.2 / 8.1 / 7.9 (C00 / C01 / C10 / C11), < 16 in the T-on cells.
- **E4:** not interpreted (spec §9 stop rule: E3 failed). Descriptively: refinement recovers y-perturbations up to σ = 0.2 but not x/shift, and it degrades perfect seeds.

Next (proposed, in order): (1) the A-only PFN as the seed slot's read-out (re-run E3, which should then pass by construction); (2) evidence-based slot weights; (3) refinement with explicit residuals (running; see the update below), then E4 again, longer training, d = 2.

## Update (2026-10-23, 13:00): refinement with explicit residuals and bounded steps

Run `results/regpfn/pilot_20261023-123300` (commit ff4d20b; `resid = true`, `max_step = 0.1`, n_iter = 8, 24 min, DONE). It got 18,300 steps, against 5,500 for the first refining run and 8,500 for read-out only, because it had the GPU alone for most of its budget. **Budgets are therefore not equal.** At n_A = 10:
- **Perfect seeds are no longer damaged.** Field error 0.004 / 0.014 / 0.013 / 0.018 (C00 / C01 / C10 / C11), against 0.006–0.066 before. With a perfect seed it beats the read-out-only model (C00 −2.90 vs −2.65; C10 −2.60 vs −2.46).
- **It has the lower NLL in every (cell, kind, σ)** (e.g. C00 y σ = 0.2: −2.04 vs −1.76; C10 shift σ = 0.2: −1.73 vs −1.62). That's partly the larger budget.
- **y-perturbations are corrected:** field error 0.20 → 0.085–0.106 at σ = 0.2. The radius is 0.2 in C00 and C10; C01 and C11 miss the 50% cut narrowly (0.101 / 0.20).
- **x / shift perturbations are still not corrected** (0.20 → 0.195; radius 0 everywhere). The model hedges instead: the null weight rises with σ to 0.42–0.66 at σ = 0.2.
- The no-evidence check holds (field change 0 at n_A = 0).

**Reading.** Explicit residuals fix the *value* part of the field and stop the drift on correct seeds. Positions don't move, because a position error leaves only an indirect trace: a y-mismatch at A's few points that, without a local slope estimate, doesn't say *which way* to move. Candidates: give the refinement the local slope of A's data and of the rendered B (finite differences between neighbours); or let the lookup compare *shapes* (the rendered B's local profile against A's points in a window); or add the front half (GW landmarks), whose job is exactly the coarse positions. Equal-budget runs come first.